# Advanced Predictions and Business Insights

This notebook is the guided entry point for the full analytics pipeline in `advanced_insights.py`. It reads `silver.csv`, creates the required `outputs/` structure, trains predictive models, explains drivers with SHAP, segments user/post behavior, detects anomalies, forecasts next-month trends, analyzes propagation networks, and writes executive-ready reports.

## Output Structure

Running the pipeline creates:

- `outputs/models/` for trained models
- `outputs/plots/` for visualizations
- `outputs/reports/` for metrics, feature importance, and business reports
- `outputs/datasets/` for scored datasets, clusters, anomalies, and forecasts

## Run the Complete Pipeline

This single cell executes the whole workflow end to end. GPU acceleration is auto-detected (`--xgboost-device auto`). If optional packages such as Prophet or SHAP are unavailable, the script uses documented fallbacks where possible and writes the status into the reports.

In [6]:
%run advanced_insights.py --data Silver.csv --output-dir . --xgboost-device cuda

2026-07-07 03:11:31,428 | INFO | Advanced analytics pipeline started
2026-07-07 03:11:31,586 | INFO | Training hardware: device=cuda cpu_cores=16 cpu_threads=16 gpu=NVIDIA GeForce RTX 2050
2026-07-07 03:11:31,587 | INFO | Current memory footprint: 616.93 MB
2026-07-07 03:11:31,587 | INFO | GPU memory: 505 MB used / 4096 MB total
2026-07-07 03:11:31,725 | INFO | Loaded Silver.csv with shape (50000, 33)
2026-07-07 03:11:31,801 | INFO | XGBoost training device: cuda (requested=cuda)
2026-07-07 03:11:31,803 | INFO | Starting XGBoost insight engine
2026-07-07 03:11:31,805 | INFO | Training XGBoost model for misinformation_probability


KeyboardInterrupt: 

## Review Executive Summary

The executive summary answers the main stakeholder questions: retained predictive models, misinformation drivers, credibility traits, user groups, anomalies, next-month trends, and strategic recommendations.

In [ ]:
from pathlib import Path

summary_path = Path("outputs/reports/executive_summary.txt")
print(summary_path.read_text(encoding="utf-8"))

Executive Summary

1. Which predictive models were retained?
The retained workflow focuses on the strongest targets: Misinformation Probability, Credibility Score. The engagement-velocity target was removed from the production pipeline because it did not achieve a stable predictive fit and offered limited additional value over the retained models.

2. What drives misinformation?
Misinformation risk is most strongly associated with risk_minus_credibility. This signal should be prioritized in moderation queues and early-warning dashboards.

3. What characterizes credible content?
Credibility is most strongly associated with sentiment_score. Stakeholders should amplify content patterns that match this profile and audit posts that move in the opposite direction.

4. Which user groups exist?
The segmentation model identified 3 K-Means user/post groups. These groups separate audiences by engagement, credibility, toxicity, and misinformation risk, enabling differentiated response strategies.


## Inspect Model Metrics

Each retained XGBoost target writes its own metrics file. These tables show predictive accuracy for misinformation probability and credibility score.

In [ ]:
import pandas as pd

metric_files = sorted(Path("outputs/reports").glob("xgboost_metrics_*.csv"))
metrics = pd.concat(
    [pd.read_csv(path).assign(model=path.stem.replace("xgboost_metrics_", "")) for path in metric_files],
    ignore_index=True,
)
metrics

,mae,rmse,r2,best_cv_rmse,model
0,0.002561,0.003683,0.999565,0.004053,credibility_score
1,0.002642,0.003847,0.999619,0.004016,misinformation_probability


## Review Segments and Anomalies

The clustered dataset contains the K-Means segment label and, when available, an HDBSCAN label. The anomalies file contains unusual records with a plain-English reason for review.

In [ ]:
clustered = pd.read_csv("outputs/datasets/clustered_data.csv")
anomalies = pd.read_csv("outputs/datasets/anomalies.csv")

display(clustered[["post_id", "kmeans_cluster", "engagement_velocity", "misinformation_probability", "credibility_score"]].head())
display(anomalies[["post_id", "anomaly_score", "anomaly_reason"]].head())

,post_id,kmeans_cluster,engagement_velocity,misinformation_probability,credibility_score
0,1.0,0,7.808,0.308,0.712
1,2.0,0,20.347,0.110,0.721
2,3.0,2,143.045,0.746,0.105
3,4.0,0,31.070,0.095,0.737
4,5.0,0,21.411,0.147,0.859


,post_id,anomaly_score,anomaly_reason
0,8609.0,-0.205692,abnormal engagement spike; high misinformation...
1,19067.0,-0.204466,abnormal engagement spike; high misinformation...
2,34279.0,-0.200668,abnormal engagement spike; high toxicity
3,49933.0,-0.193163,abnormal engagement spike; high misinformation...
4,41020.0,-0.187809,abnormal engagement spike; high misinformation...
